# Does the intelligent router actually route?

A multi-agent tutor deployed on Flowise: an LLM router in front of three specialists.

| Branch | Handles |
|---|---|
| Conceptual | theory, definitions |
| Practical | worked exercises, computations |
| Admin | deadlines, grades, submissions |

The obvious way to test it is to send a few questions and read the answers. This notebook
is about why that is not an evaluation, and what a real one needs.

Everything below imports `src/routereval/`, which is covered by the test suite and runs
with no third-party dependency.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from routereval import baseline
from routereval.cases import ADVERSARIAL, CASES, Branch
from routereval.client import Answer, FlowiseClient, ScriptedClient
from routereval.detect import detect
from routereval.evaluate import compare, evaluate_client, evaluate_function

## The problem nobody mentions: the answer does not say who wrote it

The prediction endpoint returns text. All three specialists are prompted to sound like
helpful tutors, so reading an answer and deciding which branch it "sounds like" means the
evaluator is grading its own guess and reporting the result as the router's accuracy.

There are exactly two honest sources for the route, and if neither is present the answer
is `None` — not a guess.

In [ ]:
# 1. the flow's execution trace, when the deployment returns one
traced = Answer(
    text="...",
    raw={
        "agentFlowExecutedData": [
            {"nodeLabel": "Condition Agent"},  # infrastructure, not a branch
            {"nodeLabel": "Practical"},
        ]
    },
)

# 2. an explicit tag the branch prompt emits
tagged = Answer(text="Here is the derivative... [branch: practical]")

# 3. neither
opaque = Answer(text="Here is the derivative...")

for name, answer in (("trace", traced), ("tag", tagged), ("neither", opaque)):
    d = detect(answer)
    print(f"{name:<9}{d.branch!s:<16}source={d.source}")

## The test set

Nine unambiguous cases, six adversarial ones, and two that belong to no branch at all.

The unambiguous cases are there to be passed. They are not the measurement — a set made
only of "what is X" and "compute Y" is passed by a regular expression, which is the next
cell.

In [ ]:
for case in ADVERSARIAL:
    print(f"{case.expected.value:<13}{case.probes:<45}{case.question}")

## The price of the model: a twenty-line keyword router

"The intelligent router works" is not a finding until something cheap has been tried. If a
regex scores the same, the model is buying latency and per-call cost and nothing else.

In [ ]:
keyword = evaluate_function(baseline.route, name="keyword baseline")
print(keyword.summary())

9 out of 10 on the unambiguous cases, and it falls apart on the adversarial ones. **That
gap is what the model has to buy.** An evaluation reporting only the aggregate would say
"70.6%" and hide the half that distinguishes the two routers.

## Running it offline

`ScriptedClient` stands in for the deployment. Three variants route identically; they
differ only in whether they report which branch ran.

In [ ]:
def scripted(mode):
    surface_verb_mistakes = {
        "What is a derivative, and what is the derivative of x^3?": Branch.CONCEPTUAL,
        "Explain how to compute a standard deviation.": Branch.PRACTICAL,
        "What is the grading formula for the final mark?": Branch.PRACTICAL,
    }
    answers = {}
    for case in CASES:
        chosen = surface_verb_mistakes.get(case.question, case.expected)
        if mode == "tag":
            answers[case.question] = Answer(text=f"...answer... [branch: {chosen.value}]")
        elif mode == "trace":
            answers[case.question] = Answer(
                text="...answer...",
                raw={
                    "agentFlowExecutedData": [
                        {"nodeLabel": "Condition Agent"},
                        {"nodeLabel": chosen.value.title()},
                    ]
                },
            )
        else:
            answers[case.question] = Answer(text="...answer...")
    return ScriptedClient(answers=answers)


reports = [
    keyword,
    evaluate_client(scripted("tag"), name="flow (tagged answers)"),
    evaluate_client(scripted("trace"), name="flow (execution trace)"),
    evaluate_client(scripted("none"), name="flow (no trace, no tag)"),
]
print(compare(reports))

The third and fourth flows make the same routing decisions and return the same text. One
scores 82%, the other scores zero — because an evaluation that cannot see the route is not
a weaker measurement, it is no measurement.

Note also what the comparison prints after the accuracy: with 17 cases, an 11.8% gap is a
direction and not a result. Reporting it as "the router is 12 points better" would be
overclaiming by an order of magnitude more precision than the test set supports.

## Against a live deployment

```bash
export FLOWISE_URL="https://cloud.flowiseai.com/api/v1/prediction/<flow-id>"
export FLOWISE_API_KEY="your-key"
```

Two details in the client that the straightforward `requests.post` version gets wrong:

- **Retries.** A single 502 in the middle of a run used to abort the whole evaluation and
  lose the cases that had already passed. Transient statuses are retried with backoff;
  401 and 403 fail immediately, because retrying a bad credential three times only takes
  three times as long to say so.
- **A session id.** Without one, the flow can carry conversation state between cases, and
  every case after the first depends on the ones before it.

In [ ]:
# Requires FLOWISE_URL and FLOWISE_API_KEY. Skipped if they are not set.
import os

if os.environ.get("FLOWISE_URL"):
    client = FlowiseClient.from_env(session_id="notebook-run-1")
    live = evaluate_client(client, name="flowise router")
    print(compare([keyword, live]))
else:
    print("FLOWISE_URL is not set — see the README for how to deploy the flow in flow/.")

## What this does not measure

- **Answer quality.** Whether the practical tutor's arithmetic is right is a different
  evaluation with a different test set.
- **Statistical significance.** 17 cases. A few hundred labelled questions and a
  confidence interval would be needed to defend a gap of a few points.
- **Non-determinism.** Each case is sent once, and routing at temperature > 0 is not
  deterministic. A serious version sends each case *k* times and reports the spread —
  which is precisely the measurement this test set is too small to support.

The finding worth carrying out of here is not a percentage. It is that **a deployment
which does not report its own routing decision cannot be evaluated at all**, and that the
fix costs one line in each branch prompt.